# ACIC 2016 — Sensitivity Analysis

This notebook evaluates robustness of meta-learner CATE estimators across two axes of distributional shift using the ACIC 2016 benchmark.

**Dataset**: ACIC 2016 (Atlantic Causal Inference Conference), a semi-synthetic benchmark with known potential outcomes.
- 10 instances loaded via `causallib` and split 80/20 into train/test sets
- ~4,302 observations, 58 raw covariates (expanded to ~79 after preprocessing)
- True ITE = `mu1 − mu0` is available for evaluation

**Sensitivity axes**:
- **Imbalance** (Axis 2): Treatment assignment is randomized (`T ⊥ X`) but the marginal probability `P(T=1)` is varied — from balanced (0.5) to heavily imbalanced (0.05). This isolates the effect of sample imbalance without introducing confounding.
- **Confounding** (Axis 3): The marginal rate is fixed near 0.5 but treatment assignment depends on covariates via a linear score `f(X)` scaled by `α`. Higher `α` means stronger confounding.

**Evaluation protocol**:
- For each scenario, treatment `T` and outcomes `Y` are re-generated on the training set; the test set uses the original `mu0`/`mu1`
- LightGBM hyperparameters are tuned separately for each scenario using `RandomizedSearchCV` (30 iterations, 3-fold CV)
- R-learner (NonParamDML) and DR-learner (DRLearner) use 5-fold cross-fitting for nuisance estimation

**Metrics** (computed on the test set per instance):
- **PEHE**: Precision in Estimating Heterogeneous Effects — RMSE between predicted and true ITE. Lower is better.
- **ATE Error**: Absolute difference between mean predicted ITE and mean true ITE. Lower is better.

**Models**:
- Meta-learners: S, T, X, R (NonParamDML, cv=5), DR (DRLearner, cv=5)
- Base models: LinearRegression, LightGBM (tuned), TabPFN, TabICL
- Standalone: CausalForestDML (LightGBM nuisance), CausalPFN

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
import matplotlib.pyplot as plt
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
from sklearn.metrics import mean_squared_error
import warnings
import torch
import tabpfn
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error
try:
    from causalpfn import CATEEstimator
except ImportError:
    print("CausalPFN not installed. Please install with: pip install causalpfn")
print(tabpfn.__version__)
import time
from scipy.special import expit

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error
import numpy as np

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

# CausalPFN does not support MPS
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL does not support MPS
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")

NUISANCE_CV  = 5   # K-fold cross-fitting for R- and DR-learner

# Set random seed for reproducibility
np.random.seed(42)

# Suppress warnings
warnings.filterwarnings("ignore")

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
# ── Import TabPFN 2.5 alongside 2.6 ────────────────────────────────────────
import subprocess, sys as _sys

TABPFN25_DIR = "./tabpfn_v25_install"
os.makedirs(TABPFN25_DIR, exist_ok=True)

# Install tabpfn 2.5 to isolated directory if not already present
_tabpfn25_installed = any("tabpfn" in d for d in os.listdir(TABPFN25_DIR))
if not _tabpfn25_installed:
    print("Installing TabPFN 2.5 to isolated directory...")
    subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "tabpfn==6.4.1",
        f"--target={TABPFN25_DIR}", "--quiet", "--no-deps"
    ])
    print("Done.")

# Save current tabpfn 2.6 module references
_tabpfn26_mods = {k: v for k, v in _sys.modules.items()
                  if k == "tabpfn" or k.startswith("tabpfn.")}

# Temporarily inject 2.5 path and load its classes
_sys.path.insert(0, TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]

import tabpfn as _tabpfn25
TabPFNRegressor25 = _tabpfn25.TabPFNRegressor
TabPFNClassifier25 = _tabpfn25.TabPFNClassifier
TABPFN25_VERSION = _tabpfn25.__version__

# Restore tabpfn 2.6
_sys.path.remove(TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]
_sys.modules.update(_tabpfn26_mods)

print(f"TabPFN 2.5 version: {TABPFN25_VERSION}")
print(f"TabPFN 2.6 version: {tabpfn.__version__}")

## 2. Configuration and Hyperparameter Grid

We define the sensitivity axes (`IMBALANCE_LEVELS`, `CONFOUNDING_ALPHAS`) and the LightGBM hyperparameter search space. `N_ITER = 30` RandomizedSearchCV draws are used per tuning call; `NUISANCE_CV = 5` sets the cross-fitting folds for R- and DR-learners.

In [ ]:
IMBALANCE_LEVELS = [0.5, 0.3, 0.15, 0.05]
CONFOUNDING_ALPHAS = [0.0, 0.5, 1.0, 3.0, 100]

LGBM_GRID = {
    "num_leaves": [15, 31, 63],
    "min_child_samples": [20, 50, 100],
    #"learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 500, 1000],
}

N_ITER = 30
N_EST = 1000
NUISANCE_CV = 5

## 3. Model Factories, Tuning Utilities, and Metrics

We define:
- **Unified tuning helper** (`tune_lgbm`): a single function covering all LightGBM tuning cases — regressor or classifier, pooled or per-arm — via `classifier` and `stratify` flags.
- **Model factories** (`make_reg`, `make_cls`, `make_lgbm_final`, etc.): instantiate learners with tuned parameters or wrapped TabPFN/TabICL estimators.
- **Metric functions**: `calculate_pehe` (RMSE of ITE) and `calculate_ate_error` (absolute ATE bias).
- `SENS_MODELS`: the full list of 22 models evaluated across both sensitivity axes.

In [ ]:
# ── LightGBM Tuning ──────────────────────────────────────────────────────────
def tune_lgbm(X, y, classifier=False, stratify=None, n_iter=N_ITER, seed=42):
    """Tune LGBM via RandomizedSearchCV. Returns best_params_ dict.

    - classifier = False  → LGBMRegressor, scored by neg_MSE
    - classifier = True   → LGBMClassifier, scored by neg_log_loss
    - stratify            → use StratifiedKFold on this variable (regression only);
                            if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)

    if classifier:
        # Propensity Model (classification)
        base    = LGBMClassifier(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    else:
        # Outcome Model (regression): S-, R-, and DR-learner
        base    = LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, stratify))
        else:
            # Single Treatment Arm: T- and X-learner
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle=True, random_state=seed).split(X))

    search = RandomizedSearchCV(
        base, LGBM_GRID, n_iter=n_iter, scoring=scoring,
        cv=cv, n_jobs=-1, random_state=seed,
    )
    search.fit(X, y)
    return search.best_params_

# Pseudo-Outcome Tuning: X-, R-, and DR-learner final-stage models
def make_lgbm_final(seed=42):
    """LGBM wrapped in RandomizedSearchCV for final-stage tuning on pseudo-outcomes."""
    return RandomizedSearchCV(
        LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1),
        LGBM_GRID, n_iter=N_ITER, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=seed,
    )

def make_reg(params, seed=42):
    return LGBMRegressor(random_state=seed, verbose=-1, **params)

def make_cls(params, seed=42):
    return LGBMClassifier(random_state=seed, verbose=-1, **params)


# ── Metric functions ──────────────────────────────────────────────────────────
def calculate_pehe(predicted_ite, true_ite):
    return np.sqrt(mean_squared_error(true_ite, predicted_ite))

def calculate_ate_error(predicted_ite, true_ite):
    return np.abs(predicted_ite.mean() - true_ite.mean())

SENS_MODELS = [
    'S-LinearRegression', 'S-LightGBM', 'S-TabPFN_v2.5', 'S-TabICL','S-TabPFN_v2.6',
    'T-LinearRegression', 'T-LightGBM', 'T-TabPFN_v2.5','T-TabPFN_v2.6','T-TabICL',
    'X-LinearRegression', 'X-LightGBM', 'X-TabPFN_v2.5','X-TabPFN_v2.6', 'X-TabICL',
    'R-LinearRegression', 'R-LightGBM', 'R-TabPFN_v2.5','R-TabPFN_v2.6', 'R-TabICL',
    'DR-LinearRegression', 'DR-LightGBM', 'DR-TabPFN_v2.5','DR-TabPFN_v2.6', 'DR-TabICL',
    'CausalForest',
    'CausalPFN',
]

## 4. Propensity Helper

`_lin_pred(X)` computes a fixed linear score from the first 25 covariates. This score drives the confounding axis: `P(T=1|X) = σ(α · f̃(X))`, where `f̃(X)` is the standardized score. Using a fixed linear form ensures consistent confounding signal across all 10 ACIC instances.

In [ ]:

# ── Helpers ───────────────────────────────────────────────────────────────────
def _lin_pred(X):
    """Fixed linear score propensity signal."""
    arr = X.values[:, :79]
    weights = np.array([
         1.0,  0.5, -0.5,  0.3, -0.3,
         0.4, -0.4,  0.2, -0.2,  0.6,
        -0.6,  0.1, -0.1,  0.35,-0.35,
         0.45,-0.45, 0.25,-0.25,  0.55,
        -0.55, 0.15,-0.15, 0.65,-0.65,
    ])
    return arr @ weights


## 5. Data Loading (ACIC 2016)

We load 10 ACIC 2016 instances via `causallib`, apply a fixed 80/20 train/test split (stratified by treatment), and store the potential outcomes `mu0`/`mu1` for ground-truth ITE evaluation.

In [ ]:
from collections import defaultdict
from causallib.datasets import load_acic16
from sklearn.model_selection import train_test_split

# Store results for all runs
# Structure: results[meta_learner][base_model][metric] = list of values
all_results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

n_datasets = 10  # Use all 10 ACIC 2016 instances
processed_datasets = []

print(f"Loading {n_datasets} ACIC 2016 instances via causallib...")

for i in range(1, n_datasets + 1):
    data = load_acic16(instance=i)
    X_full   = data.X.reset_index(drop=True)
    T_full   = data.a.values
    Y_full   = data.y.values
    mu0_full = data.po['0'].values
    mu1_full = data.po['1'].values
    feature_names = X_full.columns.tolist()

    # Stratified 80/20 train/test split (stratified on treatment)
    idx_train, idx_test = train_test_split(
        np.arange(len(T_full)),
        test_size=0.2,
        random_state=i,
        stratify=T_full
    )

    X_train = X_full.iloc[idx_train].reset_index(drop=True)
    X_test  = X_full.iloc[idx_test].reset_index(drop=True)
    T_train = T_full[idx_train]
    T_test  = T_full[idx_test]
    Y_train = Y_full[idx_train]
    Y_test  = Y_full[idx_test]
    mu0_test = mu0_full[idx_test]
    mu1_test = mu1_full[idx_test]
    true_ITE_test = mu1_test - mu0_test

    processed_datasets.append({
        'id':            i,
        'X_train':       X_train,
        'X_test':        X_test,
        'T_train':       T_train,
        'T_test':        T_test,
        'Y_train':       Y_train,
        'Y_test':        Y_test,
        'mu0_train':     mu0_full[idx_train],
        'mu1_train':     mu1_full[idx_train],
        'true_ITE_test': true_ITE_test,
    })

print(f"Loaded {len(processed_datasets)} instances.")
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}, Features: {X_train.shape[1]}")

## 6. Meta-Learner Factory

`_make_learner(name, ...)` instantiates any of the 22 models given pre-tuned LightGBM parameter dicts. This factory pattern decouples tuning (done once per scenario) from model construction (done once per model per scenario).

In [ ]:

def _make_learner(name, s_s, r_nuisance, s_cls, s_ctrl, s_trt):
    # ── S-learners ────────────────────────────────────────────────────────────
    if name == 'S-LinearRegression':
        return SLearner(overall_model=LinearRegression())
    elif name == 'S-LightGBM':
        return SLearner(overall_model=make_reg(s_s, 42))
    elif name == 'S-TabPFN_v2.5':
        return SLearner(overall_model=TabPFNRegressor25(device=device))
    elif name == 'S-TabPFN_v2.6':
        return Slearner(overall_model = TabPFNRegressor(device=device))
    elif name == 'S-TabICL':
        return SLearner(overall_model=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False))

    # ── T-learners ────────────────────────────────────────────────────────────
    elif name == 'T-LinearRegression':
        return TLearner(models=(LinearRegression(), LinearRegression()))
    elif name == 'T-LightGBM':
        return TLearner(models=(make_reg(s_ctrl, 42), make_reg(s_trt, 43)))
    elif name == 'T-TabPFN_v2.5':
        return TLearner(models=(TabPFNRegressor25(device=device), TabPFNRegressor25(device=device)))
    elif name == 'T-TabPFN_v2.6':
        return TLearner(models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)))
    elif name == 'T-TabICL':
        return TLearner(models=(
            TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)))

    # ── X-learners ────────────────────────────────────────────────────────────
    elif name == 'X-LinearRegression':
        return XLearner(
            models=(LinearRegression(), LinearRegression()),
            cate_models=(LinearRegression(), LinearRegression()),
            propensity_model=LogisticRegression(max_iter=1000, random_state=42))
    elif name == 'X-LightGBM':
        return XLearner(
            models=(make_reg(s_ctrl, 42), make_reg(s_trt, 43)),
            cate_models=(make_lgbm_final(46), make_lgbm_final(47)),
            propensity_model=make_cls(s_cls, 44))
    elif name == 'X-TabPFN_v2.5':
        return XLearner(
            models=(TabPFNRegressor25(device=device), TabPFNRegressor25(device=device)),
            cate_models=(TabPFNRegressor25(device=device), TabPFNRegressor25(device=device)),
            propensity_model=TabPFNClassifier25(device=device))
    elif name == 'X-TabPFN_v2.6':
        return XLearner(
            models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            cate_models=(TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            propensity_model=TabPFNClassifier(device=device))
    elif name == 'X-TabICL':
        return XLearner(
            models=(TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
                    TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)),
            cate_models=(TabICLRegressor(device=tabicl_device, random_state=44, verbose=False),
                         TabICLRegressor(device=tabicl_device, random_state=45, verbose=False)),
            propensity_model=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False))

    # ── R-learners ────────────────────────────────────────────────────────────
    elif name == 'R-LinearRegression':
        return NonParamDML(
            model_y=LinearRegression(),
            model_t=LogisticRegression(max_iter=1000, random_state=42),
            model_final=LinearRegression(),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-LightGBM':
        return NonParamDML(
            model_y=make_reg(r_nuisance, 42),    # tuned on X only — correct for R-learner
            model_t=make_cls(s_cls, 43),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-TabPFN_v2.5':
        return NonParamDML(
            model_y=TabPFNRegressor25(device=device),
            model_t=TabPFNClassifier25(device=device),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-TabPFN_v2.6':
        return NonParamDML(
            model_y=TabPFNRegressor(device=device),
            model_t=TabPFNClassifier(device=device),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)
    elif name == 'R-TabICL':
        return NonParamDML(
            model_y=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            model_t=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            model_final=make_lgbm_final(44),
            discrete_treatment=True, cv=NUISANCE_CV)

    # ── DR-learners ───────────────────────────────────────────────────────────
    elif name == 'DR-LinearRegression':
        return DRLearner(
            model_regression=LinearRegression(),
            model_propensity=LogisticRegression(max_iter=1000, random_state=42),
            model_final=LinearRegression(),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-LightGBM':
        return DRLearner(
            model_regression=make_reg(s_s, 42),  # tuned on [X,T], matches DRLearner's internal input
            model_propensity=make_cls(s_cls, 44),
            model_final=make_lgbm_final(45),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-TabPFN_v2.5':
        return DRLearner(
            model_regression=TabPFNRegressor25(device=device),
            model_propensity=TabPFNClassifier25(device=device),
            model_final=TabPFNRegressor25(device=device),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-TabPFN_v2.6':
        return DRLearner(
            model_regression=TabPFNRegressor(device=device),
            model_propensity=TabPFNClassifier(device=device),
            model_final=TabPFNRegressor(device=device),
            min_propensity=0.05, cv=NUISANCE_CV)
    elif name == 'DR-TabICL':
        return DRLearner(
            model_regression=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            model_propensity=TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            model_final=TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            min_propensity=0.05, cv=NUISANCE_CV)

    # ── CausalForest ──────────────────────────────────────────────────────────
    elif name == 'CausalForest':
        return CausalForestDML(
            model_y=LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            model_t=LGBMClassifier(random_state=43, verbose=-1, **params_prop),
            discrete_treatment=True,
            cv=NUISANCE_CV,
            n_estimators=200,
            min_samples_leaf=5,
            random_state=rs,
        )

    raise ValueError(f"Unknown model: {name}")


## 7. Scenario Generators and Sensitivity Evaluation Loop

We define two scenario generators:
- `gen_randomized(X, mu0, mu1, rng, p)`: randomized assignment at rate `p` — no confounding.
- `gen_confounded(X, mu0, mu1, rng, alpha)`: covariate-dependent assignment with strength `α`.

`run_sensitivity_analysis` iterates over all datasets × scenarios × models, tunes LightGBM once per scenario, fits each learner, and records PEHE and ATE error per row.

In [ ]:
# ── Scenario generators ───────────────────────────────────────────────────────
def gen_randomized(X, mu0, mu1, rng, p=0.5):
    """
    Axis 1 / 2: No confounding.
    T ~ Bern(p), independent of X.  P(T|X) = P(T) = p.
    Y = T·mu1 + (1-T)·mu0.
    """
    T = rng.binomial(1, p, len(X))
    Y = T * mu1 + (1 - T) * mu0
    return T, Y


def gen_confounded(X, mu0, mu1, rng, alpha=1.0):
    """
    Axis 3: Confounding with controlled imbalance.
    P(T=1|X) = sigma(alpha · f̃(X)),  f̃(X) = f(X) − mean(f(X)).
    Centering ensures E[P(T=1|X)] ≈ 0.5 for all alpha,
    so imbalance and confounding are decoupled.
    """
    lp = _lin_pred(X)
    lp = (lp - lp.mean()) / (lp.std() + 1e-8)   # standardize: E[P(T)]=0.5, scale consistent across datasets
    prop = expit(alpha * lp)
    T = rng.binomial(1, prop, len(X))
    Y = T * mu1 + (1 - T) * mu0
    return T, Y


# ── Core evaluation loop ──────────────────────────────────────────────────────
def run_sensitivity_analysis(datasets, models=SENS_MODELS, n_reps=1, seed=42):
    """
    Two-axis sensitivity analysis over all ACIC 2016 instances.

    For each instance × axis × parameter × random seed:
      - Re-generates T (and Y) under the given scenario
      - Trains a meta-learner on (X_train, T_train, Y_train)
      - Evaluates PEHE and ATE error on (X_test, true_ITE_test)

    The ground truth true_ITE_test = mu1_test − mu0_test is the same
    in every scenario — only the training difficulty changes.

    Parameters
    ----------
    datasets : list of dicts with keys:
        X_train, X_test, mu0_train, mu1_train, true_ITE_test
    models   : list of str — model names from SENS_MODELS
    n_reps   : int — random T-assignment seeds per instance per scenario
    seed     : int — master random seed

    Returns
    -------
    pd.DataFrame with columns:
        axis, param, model, rep, seed_rep, pehe, ate_error
    """
    rng_master = np.random.default_rng(seed)
    rows = []
    timings = {m: [] for m in models}  # elapsed seconds per fit, per model

    # Resolve CausalPFN device once — avoids repeated torch calls inside inner loop
    _cpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    for ds in datasets:
        rep_id = ds['id']
        X_tr, X_te     = ds['X_train'], ds['X_test']
        mu0_tr, mu1_tr = ds['mu0_train'], ds['mu1_train']
        true_ite       = ds['true_ITE_test']
        print(f"\n── Rep {rep_id:3d}/{len(datasets)} ────────────────────────────────")

        for seed_rep in range(n_reps):
            rng = np.random.default_rng(rng_master.integers(0, int(1e9)))

            # ── Axis 2: imbalance ─────────────────────────────────────────────
            for p in IMBALANCE_LEVELS:
                T_tr, Y_tr = gen_randomized(X_tr, mu0_tr, mu1_tr, rng, p=p)
                realized_rate = T_tr.mean()
                if T_tr.sum() < 5 or (1 - T_tr).sum() < 5:
                    continue   # degenerate split — skip

                # Tune LightGBM on THIS scenario's data — best model for this exact setting
                t0 = time.time()
                ctrl_mask  = T_tr == 0
                trt_mask   = T_tr == 1
                X_with_T   = np.column_stack([X_tr, T_tr])
                s_s        = tune_lgbm(X_with_T,            Y_tr, stratify=T_tr)   # S-learner outcome (X+T features)
                r_nuisance = tune_lgbm(X_tr,                Y_tr, stratify=T_tr)   # outcome nuisance (R/DR)
                s_cls      = tune_lgbm(X_tr,                T_tr, classifier=True)  # propensity model
                s_ctrl     = tune_lgbm(X_tr[ctrl_mask], Y_tr[ctrl_mask])            # T/X control arm outcome
                s_trt      = tune_lgbm(X_tr[trt_mask],  Y_tr[trt_mask])             # T/X treated arm outcome
                print(f"  LightGBM tuning: {time.time() - t0:.1f}s")

                for mname in models:
                    try:
                        t0 = time.time()
                        if mname == 'CausalPFN':
                            learner = CATEEstimator(device=_cpfn_device, verbose=False)
                            learner.fit(
                                np.asarray(X_tr, dtype=np.float32),
                                T_tr.astype(np.float32),
                                Y_tr.astype(np.float32))
                            te = np.asarray(
                                learner.estimate_cate(np.asarray(X_te, dtype=np.float32))
                            ).reshape(-1)
                        else:
                            learner = _make_learner(mname, s_s, r_nuisance, s_cls, s_ctrl, s_trt)
                            if mname == 'CausalForest':
                                learner.tune(Y_tr, T_tr, X=X_tr)   # original calls .tune() before .fit()
                            learner.fit(Y_tr, T_tr, X=X_tr)
                            te = learner.effect(X_te)
                        elapsed = time.time() - t0
                        timings[mname].append(elapsed)
                        print(f"  imb p={p:.2f} | {mname:20s} | {elapsed:.3f}s")
                        rows.append(dict(
                            axis='imbalance', param=p, model=mname,
                            rep=rep_id, seed_rep=seed_rep,
                            realized_treatment_rate=realized_rate,
                            pehe=calculate_pehe(te, true_ite),
                            ate_error=calculate_ate_error(te, true_ite),
                        ))
                    except Exception as e:
                        print(f"\n[imbalance] {mname} rep={rep_id} p={p}: {e}")

            # ── Axis 3: confounding ───────────────────────────────────────────
            for alpha in CONFOUNDING_ALPHAS:
                T_tr, Y_tr = gen_confounded(X_tr, mu0_tr, mu1_tr, rng, alpha=alpha)
                realized_rate = T_tr.mean()
                if T_tr.sum() < 5 or (1 - T_tr).sum() < 5:
                    continue

                # Tune LightGBM on THIS scenario's data — best model for this exact setting
                ctrl_mask  = T_tr == 0
                trt_mask   = T_tr == 1
                X_with_T   = np.column_stack([X_tr, T_tr])
                s_s        = tune_lgbm(X_with_T,            Y_tr, stratify=T_tr)   # S-learner outcome (X+T features)
                r_nuisance = tune_lgbm(X_tr,                Y_tr, stratify=T_tr)   # outcome nuisance (R/DR)
                s_cls      = tune_lgbm(X_tr,                T_tr, classifier=True)  # propensity model
                s_ctrl     = tune_lgbm(X_tr[ctrl_mask], Y_tr[ctrl_mask])            # T/X control arm outcome
                s_trt      = tune_lgbm(X_tr[trt_mask],  Y_tr[trt_mask])             # T/X treated arm outcome

                for mname in models:
                    try:
                        t0 = time.time()
                        if mname == 'CausalPFN':
                            learner = CATEEstimator(device=_cpfn_device, verbose=False)
                            learner.fit(
                                np.asarray(X_tr, dtype=np.float32),
                                T_tr.astype(np.float32),
                                Y_tr.astype(np.float32))
                            te = np.asarray(
                                learner.estimate_cate(np.asarray(X_te, dtype=np.float32))
                            ).reshape(-1)
                        else:
                            learner = _make_learner(mname, s_s, r_nuisance, s_cls, s_ctrl, s_trt)
                            if mname == 'CausalForest':
                                learner.tune(Y_tr, T_tr, X=X_tr)   # original calls .tune() before .fit()
                            learner.fit(Y_tr, T_tr, X=X_tr)
                            te = learner.effect(X_te)
                        elapsed = time.time() - t0
                        timings[mname].append(elapsed)
                        print(f"  cnf α={alpha:<5.1f} | {mname:20s} | {elapsed:.3f}s")
                        rows.append(dict(
                            axis='confounding', param=alpha, model=mname,
                            rep=rep_id, seed_rep=seed_rep,
                            realized_treatment_rate=realized_rate,
                            pehe=calculate_pehe(te, true_ite),
                            ate_error=calculate_ate_error(te, true_ite),
                        ))
                    except Exception as e:
                        print(f"\n[confounding] {mname} rep={rep_id} alpha={alpha}: {e}")


    # ── Timing summary ────────────────────────────────────────────────────────
    print("\n── Timing summary (mean ± std seconds per fit) ──────────────────")
    for mname in models:
        t_arr = timings[mname]
        if t_arr:
            print(f"  {mname:20s}: {np.mean(t_arr):.3f}s ± {np.std(t_arr):.3f}s  (n={len(t_arr)})")

    return pd.DataFrame(rows)




## 8. Plot Colors

In [ ]:
# Distinct colors for standalone causal models in plots
_MODEL_COLORS = {
    'CausalPFN':    '#d62728',  # red
    'CausalForest': '#9467bd',  # purple
}

## 9. Visualization

`plot_sensitivity` produces a 2×2 grid of line plots (metric × axis). Each line is one model; CausalPFN and CausalForest are highlighted with fixed colors and dashed lines.

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
def plot_sensitivity(df, save_path='sensitivity_analysis_acic.png'):
    """
    2×2 grid: rows = metrics (PEHE, ATE error), cols = axes (imbalance, confounding).
    CausalPFN and CausalForest are plotted in distinct fixed colors.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axis_meta = {
        'imbalance':   ('P(T=1)  [0.5 = balanced]',   'Imbalance  [T ⊥ X,  P(T) varies]'),
        'confounding': ('α  [0 = no confounding]',     'Confounding  [P(T) ≈ 0.5,  α varies]'),
    }

    for col, axis_key in enumerate(['imbalance', 'confounding']):
        df_ax  = df[df['axis'] == axis_key]
        xlabel, title_suffix = axis_meta[axis_key]

        for row, metric in enumerate(['pehe', 'ate_error']):
            ax = axes[row][col]
            for model in sorted(df_ax['model'].unique()):
                grp   = df_ax[df_ax['model'] == model].groupby('param')[metric]
                means = grp.mean()
                sems  = grp.sem()
                color = _MODEL_COLORS.get(model, None)
                lw    = 2.5 if model in _MODEL_COLORS else 1.8
                ls    = '--' if model in _MODEL_COLORS else '-'
                ax.errorbar(means.index, means.values, yerr=sems.values,
                            marker='o', capsize=4, linewidth=lw, linestyle=ls,
                            color=color, label=model)

            ax.set_xlabel(xlabel)
            ax.set_ylabel(metric.upper())
            ax.set_title(f'{metric.upper()} — {title_suffix}')
            ax.legend(framealpha=0.9)
            ax.grid(True, alpha=0.3)

    plt.suptitle('Sensitivity Analysis: Imbalance vs. Confounding (ACIC 2016)', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved → {save_path}")

In [ ]:
sens_df[sens_df['axis'] == 'confounding'].groupby(['param', 'rep'])['realized_treatment_rate'].mean().unstack('rep')

## 10. Run and Results

We execute the full sensitivity sweep (`n_reps=1` per instance per scenario), save results to CSV, print summary tables grouped by axis and model, and render the visualization.

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"
# ── Run ────────────────────────────────────────────────────────────────────────
# n_reps=1: each ACIC 2016 instance is evaluated with 1 independent T-assignment
# per scenario → 10 × 1 = 10 samples per scenario point.
print("Running sensitivity analysis (n_reps=1 per instance per scenario)...")
sens_df = run_sensitivity_analysis(processed_datasets, models=SENS_MODELS, n_reps=1, seed=42)
sens_df.to_csv('sensitivity_results_ihdp0404.csv', index=False)
print(f"Saved → sensitivity_results_acic.csv  ({len(sens_df)} rows)\n")

# ── Summary table ──────────────────────────────────────────────────────────────
print("── Imbalance axis (mean over all instances & seeds) ─────────────────")
print(sens_df[sens_df['axis'] == 'imbalance']
      .groupby(['param', 'model'])[['pehe', 'ate_error']]
      .mean().round(4).to_string())

print("\n── Confounding axis ────────────────────────────────────────────────────")
print(sens_df[sens_df['axis'] == 'confounding']
      .groupby(['param', 'model'])[['pehe', 'ate_error']]
      .mean().round(4).to_string())

# ── Plot ───────────────────────────────────────────────────────────────────────
plot_sensitivity(sens_df)